## AuxTel AzEl offsets - 15-Apr-21

In this notebook, investigate az-el offsets from 11-Mar-21

In [1]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.time import Time, TimeDelta
from lsst_efd_client import EfdClient
from lsst.daf.persistence import Butler
from lsst.pipe.tasks.characterizeImage import CharacterizeImageTask, CharacterizeImageConfig

In [2]:
REPO_DIR = '/project/shared/auxTel'
butler = Butler(REPO_DIR)
dayObs = '2021-03-11'
firstExpId = 2021031100346
lastExpId = 2021031100424

<ipython-input-2-0ef545bbea13>:2: FutureWarning: Gen2 Butler has been deprecated (Butler). It will be removed sometime after v23.0 but no earlier than the end of 2021.
  butler = Butler(REPO_DIR)
<ipython-input-2-0ef545bbea13>:2: FutureWarning: Gen2 Butler has been deprecated (LatissMapper). It will be removed sometime after v23.0 but no earlier than the end of 2021.
  butler = Butler(REPO_DIR)


In [3]:
# This queries the header information in the butler metadata
# This is a limited amount of information, but returns very quickly
visits = butler.queryMetadata('raw', ['expId', 'EXPTIME', 'imagetype', 'object', 'filter', 'DATE'],\
                              detector=0, dayObs=dayObs)
myVisits = []
for visit in visits:
    if (int(visit[0]) >= firstExpId) and (int(visit[0]) <= lastExpId):
        myVisits.append(visit)
        print(visit)

(2021031100346, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:09:25.819')
(2021031100347, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:09:38.545')
(2021031100348, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:09:52.002')
(2021031100349, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:10:05.546')
(2021031100350, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:10:19.047')
(2021031100351, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:10:32.595')
(2021031100352, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:10:45.298')
(2021031100353, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:10:58.350')
(2021031100354, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:11:12.509')
(2021031100355, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:11:26.707')
(2021031100356, 5.0, 'SKYEXP', 'HD 125371', 'RG610~empty', '2021-03-12T04:11:40.856')
(2021031100357, 5.0, 'SKYEXP', 'HD 125371', 'RG610~emp

In [4]:
# Get EFD client
client = EfdClient('summit_efd')

In [32]:
# These are for finding the timestamps of the offset events
backUp = 5 # seconds before first image to get initial offset
start = Time(myVisits[0][5],scale='tai') - TimeDelta(backUp, format='sec')
end = Time(myVisits[-1][5],scale='tai')
timestamp = f"time >= {start} AND time <= {end}"

In [33]:
# Now get the offsets applied
offsets = await client.select_time_series("lsst.sal.ATPtg.command_offsetAzEl", ['*'],
                                          start, end)

In [34]:
print(len(offsets), offsets.columns)

77 Index(['az', 'el', 'num', 'private_host', 'private_identity',
       'private_kafkaStamp', 'private_origin', 'private_rcvStamp',
       'private_revCode', 'private_seqNum', 'private_sndStamp'],
      dtype='object')


In [35]:
# Now get the offsets applied
offsets = await client.select_time_series("lsst.sal.ATPtg.command_azElTarget", ['*'],
                                          start, end)

In [36]:
print(len(offsets), offsets.columns)

0 Index([], dtype='object')


In [52]:
# Now get the offsets applied
offsets = await client.select_time_series("lsst.sal.ATPtg.command_poriginOffset", ['*'],
                                          start, end)

In [55]:
print(len(offsets), offsets.columns, offsets['dy'])

1 Index(['dx', 'dy', 'num', 'private_host', 'private_identity',
       'private_kafkaStamp', 'private_origin', 'private_rcvStamp',
       'private_revCode', 'private_seqNum', 'private_sndStamp'],
      dtype='object') 2021-03-12 04:28:31.338000+00:00    9.364872
Name: dy, dtype: float64


In [37]:
# Now get the offsets applied
offsets = await client.select_time_series("lsst.sal.ATPtg.command_guidingAndOffsets", ['*'],
                                          start, end)

In [38]:
print(len(offsets), offsets.columns)

0 Index([], dtype='object')


In [50]:
# Now get the offsets applied
offsets = await client.select_time_series("lsst.sal.ATPtg.currentTargetStatus", ['*'],
                                          start, end)

In [51]:
print(len(offsets), offsets.columns, offsets['demandEl'])

8614 Index(['airmass', 'demandAz', 'demandAzVelocity', 'demandDec', 'demandEl',
       'demandElVelocity', 'demandRa', 'demandRot', 'demandRotVelocity', 'ha',
       'parAngle', 'private_host', 'private_identity', 'private_kafkaStamp',
       'private_origin', 'private_rcvStamp', 'private_revCode',
       'private_seqNum', 'private_sndStamp', 'timestamp'],
      dtype='object') 2021-03-12 04:09:20.923000+00:00    30.324877
2021-03-12 04:09:21.123000+00:00    30.324888
2021-03-12 04:09:21.323000+00:00    30.324898
2021-03-12 04:09:21.523000+00:00    30.324907
2021-03-12 04:09:21.723000+00:00    30.324917
                                      ...    
2021-03-12 04:38:03.843000+00:00    30.403902
2021-03-12 04:38:04.043000+00:00    30.403912
2021-03-12 04:38:04.243000+00:00    30.403921
2021-03-12 04:38:04.444000+00:00    30.403930
2021-03-12 04:38:04.644000+00:00    30.403940
Name: demandEl, Length: 8614, dtype: float64


In [41]:
# Now get the offsets applied
offsets = await client.select_time_series("lsst.sal.ATPtg.logevent_currentTarget", ['*'],
                                          start, end)

In [42]:
print(len(offsets), offsets.columns)

0 Index([], dtype='object')


In [43]:
# Now get the offsets applied
offsets = await client.select_time_series("lsst.sal.ATPtg.namedAzEl", ['*'],
                                          start, end)

In [47]:
print(len(offsets), offsets.columns, offsets['elPositions'])

345 Index(['azPositions', 'elPositions', 'names', 'private_host',
       'private_identity', 'private_kafkaStamp', 'private_origin',
       'private_rcvStamp', 'private_revCode', 'private_seqNum',
       'private_sndStamp', 'rotPositions'],
      dtype='object') 2021-03-12 04:09:22.378000+00:00    80,90
2021-03-12 04:09:27.381000+00:00    80,90
2021-03-12 04:09:32.395000+00:00    80,90
2021-03-12 04:09:37.387000+00:00    80,90
2021-03-12 04:09:42.388000+00:00    80,90
                                    ...  
2021-03-12 04:37:43.487000+00:00    80,90
2021-03-12 04:37:48.487000+00:00    80,90
2021-03-12 04:37:53.490000+00:00    80,90
2021-03-12 04:37:58.494000+00:00    80,90
2021-03-12 04:38:03.495000+00:00    80,90
Name: elPositions, Length: 345, dtype: object


In [ ]:
# Plot all of it
startPlot = Time('2021-03-12T04:09:00') #this is UTC
endPlot = Time('2021-03-13T04:20:00')

fig = plt.figure()
plt.suptitle(f"Offsets - 11-Mar-21", fontsize = 18)
# Azimuth axis
plt.subplot(1,1,1)
#ax1 = az['azimuthCalculatedAngle'].plot(color='red', label='azimuth')
ax1.set_ylim(0,1.0)
for i in range(len(offsets)):
    t = Time(offsets.index[i]).tai.isot
    ax1.axvline(t, ymin=0.5, ymax=1.0, color="blue", linestyle="--")


In [ ]:
ax1.axvline(t, ymin=0.8, ymax=0.9, color="blue", linestyle="--", label='inPosition')
for i in range(len(startTracking)):
    t = Time(startTracking.index[i]).tai.isot
    ax1.axvline(t, ymin=0.7, ymax=0.8, color="green", linestyle="--")
ax1.axvline(t, ymin=0.7, ymax=0.8, color="green", linestyle="--", label='startTracking')
for i in range(len(stopTracking)):
    t = Time(stopTracking.index[i]).tai.isot
    ax1.axvline(t, ymin=0.6, ymax=0.7, color="red", linestyle="--")
ax1.axvline(t, ymin=0.6, ymax=0.7, color="red", linestyle="--", label='stopTracking')
for i in range(len(commandStart)):
    t = Time(commandStart.index[i]).tai.isot
    ax1.axvline(t, ymin=0.5, ymax=0.6, color="cyan", linestyle="--")
ax1.axvline(t, ymin=0.5, ymax=0.6, color="cyan", linestyle="--", label='commandStart')

firstBias = '2021-03-10T22:31:37' # Got this from the header
t = Time(firstBias).tai.isot
ax1.axvline(t, ymin=0.9, ymax=1.0, color="magenta", linestyle="--", label='firstBias')

firstObject = '2021-03-11T00:13:19' # Got this from the header
t = Time(firstObject).tai.isot
ax1.axvline(t, ymin=0.9, ymax=1.0, color="magenta", linestyle="--", label='firstObject')

ax1.set_xlim(startPlot.tai.isot,endPlot.tai.isot)
ax1.legend()#loc='bottom')


In [ ]:
# Now append the applied offsets to the list of visits
fullVisits = []
for i, visit in enumerate(myVisits):
    if i == 0:
        startTime = Time(myVisits[i][5],scale='tai',precision=0) - TimeDelta(backUp, format='sec')
        startTime = startTime.isot
    else:
        startTime = Time(myVisits[i][5],scale='tai',precision=0).isot
    if i == len(myVisits) - 1:
        endTime = Time(myVisits[i][5],scale='tai',precision=0) + TimeDelta(backUp, format='sec')
    else:
        endTime = Time(myVisits[i+1][5],scale='tai',precision=0).isot
    try:
        offset = offsets.loc[startTime:endTime].values
        if len(offset) == 1:
            newList = list(visit)
            newList.append(offset[0][0])
            newList.append(offset[0][1])
            fullVisits.append(newList)
        else:
            continue
    except:
        continue


In [ ]:
len(fullVisits)

In [ ]:
# The last two values are the applied offsets in az and el in arcseconds
fullVisits[0]

In [ ]:
# This way goes directly to the fits files without using the butler
import glob
files = glob.glob(REPO_DIR+'/_parent/raw/2021-03-11/*')
print(len(files))
hduList = pf.open(files[346])
hdr = hduList[0].header
for key in hdr.keys():
    print(key, hdr[key])

In [ ]:
REPO_DIR = '/project/shared/auxTel'
butler = Butler(REPO_DIR)

test = butler.get('raw', dataId={'expId': 2021031100346, 'detector': 0, 'dayObs': '2021-03-11'})